In [ ]:
from pathlib import Path
import os, sys
if (Path.cwd() / "AGENTS.md").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "AGENTS.md").exists():
    ROOT = Path.cwd().parent
else:
    ROOT = Path.home() / "hibah-riset"
DATA, EXT, EXP = ROOT / "data" / "s2", ROOT / "external", ROOT / "experiments" / "s2_tracker"
for d in (DATA, EXP):
    d.mkdir(parents=True, exist_ok=True)
os.environ.update(S2_ROOT=str(ROOT), S2_DATA=str(DATA), S2_EXT=str(EXT), S2_EXP=str(EXP))
print("ROOT :", ROOT)
print("DATA :", DATA)
print("python:", sys.executable)

# 70 — Evaluasi TrackEval (HOTA / IDF1 / MOTA / IDSW / Frag)

Kernel: `s2-main`.

Menggunakan **Python API** TrackEval (bukan CLI) agar hasil terurai jadi dict langsung.
Catatan jujur: `DO_PREPROC=False` (tanpa penghapusan distractor) — angka tidak persis sama
dengan leaderboard MOTChallenge, tapi **konsisten antar tracker** → valid untuk perbandingan
DiffMOT vs OC-SORT (tujuan Skenario B).

In [ ]:
from pathlib import Path
import os, sys
if (Path.cwd() / "AGENTS.md").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "AGENTS.md").exists():
    ROOT = Path.cwd().parent
else:
    ROOT = Path.home() / "hibah-riset"
DATA, EXT, EXP = ROOT / "data" / "s2", ROOT / "external", ROOT / "experiments" / "s2_tracker"
for d in (DATA, EXP):
    d.mkdir(parents=True, exist_ok=True)
os.environ.update(S2_ROOT=str(ROOT), S2_DATA=str(DATA), S2_EXT=str(EXT), S2_EXP=str(EXP))
print("ROOT :", ROOT)
print("DATA :", DATA)
print("python:", sys.executable)

In [ ]:
import sys
sys.path.insert(0, str(EXT / "TrackEval"))
import trackeval
print("trackeval OK", trackeval.__file__)

### Susun folder hasil ala TrackEval

In [ ]:
import shutil
from pathlib import Path
def organize(results_dir, trackers_root, tracker_name, ds):
    dst = trackers_root / ds / tracker_name / "data"
    dst.mkdir(parents=True, exist_ok=True)
    for f in (results_dir).glob("*.txt"):
        shutil.copy2(f, dst / f.name)
    return len(list(dst.glob("*.txt")))

trackers_root = EXP / "trackeval_trackers"
for ds, srcs in [("mot20", [("diffmot", EXP/"diffmot_results"/"mot20"), ("ocsort", EXP/"ocsort_results"/"mot20")]),
                 ("dance", [("diffmot", EXP/"diffmot_results"/"dancetrack"), ("ocsort", EXP/"ocsort_results"/"dancetrack")])]:
    for name, src in srcs:
        n = organize(src, trackers_root, name, ds)
        print(ds, name, n, "files")

### Tulis seqmap (dinamis dari folder GT — tidak mengasumsikan file dari repo)

In [ ]:
def write_seqmap(gt_split_root: Path, seqmap_path: Path):
    seqs = sorted(p.name for p in gt_split_root.iterdir() if p.is_dir() and (p/"gt"/"gt.txt").exists())
    seqmap_path.parent.mkdir(parents=True, exist_ok=True)
    seqmap_path.write_text("name\n" + "\n".join(seqs) + "\n")
    print(seqmap_path, len(seqs), "sekuens")
    return seqs

seqs_mot20 = write_seqmap(DATA/"mot20"/"train", DATA/"seqmaps"/"MOT20-train.txt")
seqs_dance = write_seqmap(DATA/"dancetrack"/"val", DATA/"seqmaps"/"dancetrack-val.txt")

### Jalankan evaluasi (per tracker, per dataset)

In [ ]:
import trackeval

def run_eval(bench, gt_folder, trackers_folder, tracker, seqmap, split, skip_split_fol):
    eval_cfg = trackeval.Evaluator.get_default_eval_config()
    eval_cfg["USE_PARALLEL"] = False
    eval_cfg["NUM_PARALLEL_CORES"] = 8
    eval_cfg["PLOT_CURVES"] = False
    eval_cfg["DISPLAY_LESS_PROGRESS"] = True
    ds_cfg = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
    ds_cfg.update(dict(
        BENCHMARK=bench, GT_FOLDER=str(gt_folder), TRACKERS_FOLDER=str(trackers_folder),
        TRACKERS_TO_EVAL=[tracker], SEQMAP_FILE=str(seqmap), SPLIT_TO_EVAL=split,
        SKIP_SPLIT_FOL=skip_split_fol, DO_PREPROC=False,
    ))
    metrics = [trackeval.metrics.HOTA(), trackeval.metrics.CLEAR(), trackeval.metrics.Identity()]
    evaluator = trackeval.Evaluator(eval_cfg)
    out = evaluator.evaluate([trackeval.datasets.MotChallenge2DBox(ds_cfg)], metrics)
    return out

def extract(out):
    rows = []
    for ds_name, trackers in out.items():
        for trk, classes in trackers.items():
            for cls, res in classes.items():
                comb = res.get("COMBINED_SEQ", {})
                def g(m, k):
                    try: return comb[m][k]
                    except Exception: return None
                rows.append(dict(dataset=ds_name, tracker=trk, cls=cls,
                                 HOTA=g("HOTA","HOTA"), MOTA=g("CLEAR","MOTA"),
                                 IDF1=g("Identity","IDF1"), IDSW=g("Identity","IDSW"),
                                 Frag=g("Identity","Frag")))
    return rows

import pandas as pd
all_rows = []
for ds_key, bench, gt, trk_root, seqmap, split, skip in [
    ("mot20", "MOT20", DATA/"mot20", trackers_root/"mot20", DATA/"seqmaps"/"MOT20-train.txt", "train", False),
    ("dance", "MOT20", DATA/"dancetrack", trackers_root/"dance", DATA/"seqmaps"/"dancetrack-val.txt", "val", True),
]:
    for tracker in ["diffmot", "ocsort"]:
        print("eval", ds_key, tracker, "...")
        out = run_eval(bench, gt, trk_root, tracker, seqmap, split, skip)
        all_rows += extract(out)
df = pd.DataFrame(all_rows)
df.to_csv(EXP / "eval_results.csv", index=False)
display(df)

**Lanjut**: `80_s2_analyze.ipynb` — tabel pembanding, plot, bahan laporan.